# 01. Multi-Agent 시스템 설계
**주제:** AI 기술 발전과 산업별 활용 사례를 분석하는 보고서 시스템

**학습 목표:** 요구사항을 업무로 분해하고, 전문 Agent 다섯 개의 책임·입출력·도구를 정한 뒤 Sequential 및 Hybrid 구조를 설계한다. LLM 클라이언트는 02·03번과 동일하게 초기화하지만 이 Notebook에서는 실제로 호출(`invoke`)하지 않는다.


## 1. 핵심 개념과 요구사항
Workflow는 정해진 단계의 실행 순서다. 전문 Agent는 목표와 입력을 받아 자기 책임 안에서 자료 선택, 도구 사용 또는 결과 검토 같은 판단을 수행한다. 여기서는 상위 Graph가 협업 순서를 정하고, Research Agent의 Subgraph가 조회 범위를 결정한 뒤 로컬 도구를 사용한다.

요구사항: **AI 기술 동향과 산업별 활용 사례를 분석하여, 출처를 제시하는 Markdown 보고서를 작성한다.** 산출물에는 기술·산업 분석과 출처·논리 점검 결과가 있어야 한다.


## 2. 전체 아키텍처
```mermaid
flowchart TD
  A[START] --> B[Research Agent]
  B --> C[Technology Analyst]
  C --> D[Industry Analyst]
  D --> E[Report Writer]
  E --> F[Review Agent]
  F --> G[END]
```
위 구조는 순차형 설계 초안이다. 기술·산업 분석은 같은 Research 결과만 읽으므로 서로 독립적이다. 03번 Notebook에서는 두 분석을 병렬로 실행한다.


## 3. Import와 LLM 설정
`.env`의 `OPENAI_API_KEY`를 사용하는 `common.ai_model.get_llm_model()`로 LLM 클라이언트를 초기화한다. 모델명은 `.env`의 `LLM_MODEL` 값을 따른다. 이 Notebook은 설계 실습이라 아래에서 `invoke`는 호출하지 않는다. 실제 분석 호출은 02번부터 실행된다.


In [1]:
from typing import TypedDict

from common.ai_model import get_llm_model

# 02·03번과 동일한 방식으로 클라이언트만 준비한다. 이 설계 노트에서는 invoke를 호출하지 않는다.
llm_model = get_llm_model()
print(llm_model.model_name)

gpt-4o-mini


## 4. 샘플 요청과 업무 분해
산출물에서 역으로 필요한 책임을 정한다. 수집 → 기술 분석·산업 분석 → 작성 → 검토 순서로 나누면, 각 Agent가 왜 필요한지 설명할 수 있다.


In [5]:
# 산출물(보고서)에서 거꾸로 필요한 업무를 먼저 나열해 본다. Agent 설계표(5번)의 입력이 된다.
user_request = "AI 기술 동향과 산업별 활용 사례를 분석하고 출처가 있는 보고서를 작성하라."
work_items = ["출처 수집", "기술 분석", "산업 분석", "보고서 작성", "근거 검토"]
print(user_request)
print("업무:", " → ".join(work_items))


AI 기술 동향과 산업별 활용 사례를 분석하고 출처가 있는 보고서를 작성하라.
업무: 출처 수집 → 기술 분석 → 산업 분석 → 보고서 작성 → 근거 검토


## 5. Agent 설계표
| Agent | 목표·책임 | 도구 | 입력 | 출력 |
|---|---|---|---|---|
| Research Agent | 요청에 맞는 자료를 고르고 출처 ID 관리 | 로컬 자료 검색 | `user_request` | `research_results` |
| Technology Analyst | 제공 자료 안에서 기술 흐름 해석 | LLM | `research_results` | `technology_analysis` |
| Industry Analyst | 산업 적용과 한계 해석 | LLM | `research_results` | `industry_analysis` |
| Report Writer | 두 분석을 종합하고 인용 ID 부착 | LLM | 두 분석·자료 | `final_report` |
| Review Agent | 인용 ID 확인과 주장·근거 대조 | 정규식·LLM | 보고서·자료 | `review_notes` |

모든 Agent가 도구를 호출할 필요는 없다. Research는 조회 도구가 필요하고, Review는 인용 ID 검사에 Python 코드를 사용한다.


## 6. 일반 Node와 전문 Agent의 차이
두 함수 모두 Graph의 Node가 될 수 있다. 전문 Agent는 역할에 맞춰 **어떤 자료를 찾을지 결정하고 도구를 사용한다**는 점이 다르다. 아래 코드는 설계 개념을 보여 주는 작은 예시이며 실제 보고서 Graph는 02번에서 만든다.


In [6]:
def fixed_workflow_node(state: dict) -> dict:
        # Workflow Node: 입력 내용과 무관하게 고정된 규칙만 적용한다(판단 없음).
        return {"summary": "AI 보고서: " + state["user_request"]}


def research_agent_step(state: dict) -> dict:
        # 전문 Agent: 요청 내용을 보고 어떤 자료를 조회할지 스스로 판단해 정한다.
        categories = ["technology", "industry"] if "산업" in state["user_request"] else ["technology"]
        return {"selected_categories": categories}


print(fixed_workflow_node({"user_request": user_request}))
print(research_agent_step({"user_request": user_request}))


{'summary': 'AI 보고서: AI 기술 동향과 산업별 활용 사례를 분석하고 출처가 있는 보고서를 작성하라.'}
{'selected_categories': ['technology', 'industry']}


## 7. 공통 State 설계
State는 Agent 간 전달 계약이다. 각 Node는 자신이 책임지는 Key만 업데이트한다. `research_results`는 출처 ID·제목·URL·근거 요약을 담은 카드 목록이다.


In [ ]:
# total=False: 실행 중 각 Agent가 자신이 맡은 Key만 채우므로, 처음부터 모든 Key가 있을 필요는 없다.
class ReportState(TypedDict, total=False):
        user_request: str
        research_results: list[dict]
        technology_analysis: str
        industry_analysis: str
        final_report: str
        review_notes: str

# 어떤 Agent가 어떤 State Key의 소유자인지 명시해 책임 경계를 분명히 한다.
state_owners = {
    "research_results": "Research Agent",
    "technology_analysis": "Technology Analyst",
    "industry_analysis": "Industry Analyst",
    "final_report": "Report Writer",
    "review_notes": "Review Agent",
}

# 
for key, owner in state_owners.items():
    print(f"{key:22} ← {owner}")

research_results       ← Research Agent
technology_analysis    ← Technology Analyst
industry_analysis      ← Industry Analyst
final_report           ← Report Writer
review_notes           ← Review Agent


## 8. 아키텍처 설계와 협업 흐름
**Sequential:** Research → Technology → Industry → Writer → Review. 설명이 쉽지만 독립 분석도 기다린다.

**Hybrid:** Research → (Technology ∥ Industry) → Writer → Review. 두 분석은 동일한 자료를 읽고 서로 다른 Key에 쓰므로 충돌이 없다. `Writer`는 둘 다 완료된 뒤 실행한다.

```mermaid
flowchart TD
  R[Research] --> T[Technology]
  R --> I[Industry]
  T --> W[Writer]
  I --> W
  W --> V[Review]
```


## 9. 핵심 개념 정리와 미션
책임 경계, State Key의 소유자, 병렬 가능 여부를 먼저 정하면 Graph 코드가 단순해진다. 출처 ID가 있다는 사실만으로 주장이 입증되지는 않는다.

**미션:** 교육·금융·의료 중 한 분야를 골라 Agent 설계표의 목표와 입출력을 바꾼다.  
기술 분석과 산업 분석이 서로의 결과를 읽어야 하는지 판단하고, 선택한 Edge를 설명한다.
